# 模型配置的动态切换

In [ ]:
from langgraph_python.core.config import anthropic_settings
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage

# 创建模型时，可以指定哪些字段可以在运行时重新配置
model = init_chat_model(
    model_provider="anthropic",
    model=anthropic_settings.default_model,
    configurable_fields=[
        "model", 
        "temperature",
        "top_p",
        "thinking"
    ]
)

In [ ]:
from langchain_core.callbacks import BaseCallbackHandler


class ConfigPrinter(BaseCallbackHandler):
    def on_chat_model_start(self, serialized, messages, **kwargs):
        params = kwargs["invocation_params"]
        print("本次调用使用的模型配置：")
        print(f"  Model:       {params.get('model')}")
        print(f"  Temperature: {params.get('temperature')}")
        print(f"  Top P:       {params.get('top_p')}")
        print(f"  Thinking:    {params.get('thinking')}")
        print(f"  Top K:       {params.get('top_k')}")


await model.ainvoke(
    input=[HumanMessage("你好")],
    config={
        "callbacks": [ConfigPrinter()]
    }
)

In [ ]:
await model.ainvoke(
    [HumanMessage("你好")],
    config={
        "configurable": {
            "model": "qwen3.7-max",
            "temperature": 0.5,
            "top_p": 0.9,
            "thinking": {"type": "disabled"},
            "top_k": 50,
        },
        "callbacks": [ConfigPrinter()],
    }
)

In [ ]:
from langgraph_python.graphs.core_agent_graph import build_graph

graph = build_graph().compile()

await graph.ainvoke(
    input={
        "messages": [HumanMessage("你好")]
    },
    config={
        "configurable": {
            "model": "qwen3.7-max",
            "temperature": 0.5,
            "top_p": 0.9,
            "thinking": {"type": "disabled"},
            "top_k": 50,
        },
        "callbacks": [ConfigPrinter()],
    }
)